In [23]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv("../instituciones_clasificadas_ok.csv")

df["Cantidad"] = pd.to_numeric(df["Cantidad"], errors="coerce").fillna(0).astype(int)

# ── Mapeo explícito de variantes → nombre canónico ──────────────────────────
PAIS_MAP = {
    # México
    "mexico": "México",
    "méxico": "México",
    # Ecuador
    "ecuador": "Ecuador",
    "ecuadoe": "Ecuador",
    "ecaudor": "Ecuador",
    "quito": "Ecuador",          # ciudad, no país
    # Egipto → era Ecuador según indicación
    "egipto": "Ecuador",
    # República Dominicana
    "dominican republic": "República Dominicana",
    "republica dominicana": "República Dominicana",
    "república dominicana": "República Dominicana",
    # Perú
    "peru": "Perú",
    "perú": "Perú",
    # Colombia
    "colombia": "Colombia",
    # Argentina
    "argentina": "Argentina",
    # Chile
    "chile": "Chile",
    # Bolivia
    "bolivia": "Bolivia",
    # Venezuela
    "venezuela": "Venezuela",
    # Uruguay
    "uruguay": "Uruguay",
    # Paraguay
    "paraguay": "Paraguay",
    # Brasil
    "brasil": "Brasil",
    "brazil": "Brasil",
    # Costa Rica
    "costa rica": "Costa Rica",
    # Guatemala
    "guatemala": "Guatemala",
    # Honduras
    "honduras": "Honduras",
    # El Salvador
    "el salvador": "El Salvador",
    # Nicaragua
    "nicaragua": "Nicaragua",
    # Panamá
    "panama": "Panamá",
    "panamá": "Panamá",
    # Cuba
    "cuba": "Cuba",
    # Puerto Rico
    "puerto rico": "Puerto Rico",
    # España
    "espana": "España",
    "españa": "España",
    "spain": "España",
}

# Conjunto de valores que NO son un país válido (ciudades, textos genéricos, etc.)
NO_ES_PAIS = {"quito", "cairo", ""}

def normalizar_pais(row):
    """
    Usa la columna 'País'. Si está vacía o no es un país reconocido,
    cae a 'País de Origen'.
    """
    def limpiar(val):
        if pd.isna(val):
            return ""
        return str(val).strip().lower()

    raw = limpiar(row.get("País", ""))

    # Si el valor está vacío, es una ciudad conocida o no tiene mapeo -> usar País de Origen
    if raw == "" or raw in NO_ES_PAIS or raw not in PAIS_MAP:
        raw = limpiar(row.get("País de Origen", ""))

    return PAIS_MAP.get(raw, str(row.get("País de Origen", raw)).strip().title())

df["pais"] = df.apply(normalizar_pais, axis=1)

PALETTE = ["#B5C9F0", "#F0B5C9", "#7FA8E8", "#E87FA8", "#C4D8F7", "#F7C4D8", "#4A86D8", "#D84A86"]

In [24]:
by_pais = df.groupby("pais")["Cantidad"].sum().reset_index()
by_pais = by_pais.sort_values("Cantidad", ascending=True)

fig = px.bar(
    by_pais, x="Cantidad", y="pais", orientation="h",
    title="Total de respuestas por país",
    labels={"Cantidad": "Respuestas", "pais": "País"},
    color="Cantidad",
    color_continuous_scale=["#C4D8F7", "#4A86D8"],
    text="Cantidad"
)
fig.update_traces(textposition="outside")
fig.update_layout(
    height=600, coloraxis_showscale=False,
    plot_bgcolor="white", paper_bgcolor="white",
    title_font_size=18
)
fig.show()

In [25]:
total_inst = df["institucion"].nunique()

print("=" * 50)
print(f"  TOTAL DE INSTITUCIONES DISTINTAS: {total_inst}")
print("=" * 50)

conteo_por_tipo = (
    df.groupby("tipo_institucion")["institucion"]
    .nunique()
    .reset_index()
    .rename(columns={
        "tipo_institucion": "Tipo de institución",
        "institucion": "Instituciones distintas"
    })
    .sort_values("Instituciones distintas", ascending=False)
)

print("\nINSTITUCIONES DISTINTAS POR TIPO:")
print("-" * 50)
print(conteo_por_tipo.to_string(index=False))

  TOTAL DE INSTITUCIONES DISTINTAS: 156

INSTITUCIONES DISTINTAS POR TIPO:
--------------------------------------------------
                  Tipo de institución  Instituciones distintas
Centro de Investigacion / Laboratorio                       56
                    Universidad / IES                       44
              Organismo Gubernamental                       15
                                 Otra                       15
                      ONG / Fundacion                        6
                  Empresa / Industria                        3
                   Hospital / Clinica                        1
                        Independiente                        1
      Instituto Tecnico / Tecnologico                        1


In [26]:
total_inst = df["institucion"].nunique()

conteo_por_tipo = (
    df.groupby("tipo_institucion")["institucion"]
    .nunique()
    .reset_index()
    .rename(columns={
        "tipo_institucion": "Tipo de institución",
        "institucion": "Instituciones distintas"
    })
    .sort_values("Instituciones distintas", ascending=True)
)

fig = px.bar(
    conteo_por_tipo,
    x="Instituciones distintas",
    y="Tipo de institución",
    orientation="h",
    title=f"Instituciones distintas por tipo  (Total: {total_inst})",
    color="Instituciones distintas",
    color_continuous_scale=["#C4D8F7", "#4A86D8"],
    text="Instituciones distintas"
)
fig.update_traces(textposition="outside")
fig.update_layout(
    height=700,
    coloraxis_showscale=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    title_font_size=18,
    margin=dict(r=80)
)
fig.show()

In [27]:
cantidad_por_tipo = (
    df.groupby("tipo_institucion")["Cantidad"]
    .sum()
    .reset_index()
    .rename(columns={"tipo_institucion": "Tipo de institución", "Cantidad": "Total preguntas aportadas"})
    .sort_values("Total preguntas aportadas", ascending=False)
)

print("=" * 55)
print("  TOTAL DE PREGUNTAS POR TIPO DE INSTITUCIÓN")
print("=" * 55)
print(cantidad_por_tipo.to_string(index=False))
print(f"\nTOTAL GENERAL: {cantidad_por_tipo['Total preguntas aportadas'].sum()}")

  TOTAL DE PREGUNTAS POR TIPO DE INSTITUCIÓN
                  Tipo de institución  Total preguntas aportadas
Centro de Investigacion / Laboratorio                        817
                  Empresa / Industria                        783
                    Universidad / IES                        555
                        Independiente                        369
                                 Otra                        198
              Organismo Gubernamental                        182
                      ONG / Fundacion                         63
                   Hospital / Clinica                          1
      Instituto Tecnico / Tecnologico                          1

TOTAL GENERAL: 2969


In [28]:
cantidad_por_tipo = (
    df.groupby("tipo_institucion")["Cantidad"]
    .sum()
    .reset_index()
    .rename(columns={"tipo_institucion": "Tipo de institución", "Cantidad": "Total preguntas aportadas"})
    .sort_values("Total preguntas aportadas", ascending=True)
)

fig = px.bar(
    cantidad_por_tipo,
    x="Total preguntas aportadas",
    y="Tipo de institución",
    orientation="h",
    title="Total de preguntas benchmark LatamGPT por tipo de institución",
    color="Total preguntas aportadas",
    color_continuous_scale=["#C4D8F7", "#4A86D8"],
    text="Total preguntas aportadas"
)
fig.update_traces(textposition="outside")
fig.update_layout(
    height=700,
    coloraxis_showscale=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    title_font_size=18,
    margin=dict(r=80)
)
fig.show()

In [29]:
cantidad_por_tipo = (
    df.groupby("tipo_institucion")["Cantidad"]
    .sum()
    .reset_index()
    .rename(columns={"tipo_institucion": "Tipo de institución", "Cantidad": "Total preguntas aportadas"})
    .sort_values("Total preguntas aportadas", ascending=False)
)

# Agrupar tipos minoritarios en "Otros"
TOP_N = 5
top = cantidad_por_tipo.head(TOP_N).copy()
otros = cantidad_por_tipo.iloc[TOP_N:]

otros_row = pd.DataFrame([{
    "Tipo de institución": f"Otros ({len(otros)} tipos)",
    "Total preguntas aportadas": otros["Total preguntas aportadas"].sum()
}])
pie_df = pd.concat([top, otros_row], ignore_index=True)

total = pie_df["Total preguntas aportadas"].sum()
pie_df["pct_str"] = (pie_df["Total preguntas aportadas"] / total * 100).map("{:.3f}%".format)

# Anotación con desglose de "Otros"
otros_lines = "".join(
    f"• {row['Tipo de institución']} ({row['Total preguntas aportadas'] / total * 100:.3f}%)<br>"
    for _, row in otros.iterrows()
)
otros_annotation = f"<b>Otros incluye:</b><br>{otros_lines}"

fig = go.Figure(go.Pie(
    labels=pie_df["Tipo de institución"],
    values=pie_df["Total preguntas aportadas"],
    hole=0.4,
    textposition="outside",
    texttemplate="%{customdata[0]}<br>%{label}",
    customdata=pie_df[["pct_str"]].values,
    hovertemplate="<b>%{label}</b><br>%{percent:.3%} · %{value} preguntas<extra></extra>",
    textfont=dict(size=11),
    showlegend=False,
))

fig.update_layout(
    title=dict(text="Distribución de preguntas por tipo de institución", font=dict(size=18)),
    height=600,
    paper_bgcolor="white",
    piecolorway=PALETTE,
    margin=dict(l=80, r=80, t=120, b=80),
    annotations=[dict(
        text=otros_annotation.rstrip("<br>"),
        x=1.2, y=1.18,
        xref="paper", yref="paper",
        showarrow=False,
        align="left",
        bgcolor="#f9f9f9",
        bordercolor="#ccc",
        borderwidth=1,
        borderpad=6,
        font=dict(size=10, color="#555"),
    )]
)
fig.show()


In [30]:
# Normalizar país de origen usando PAIS_MAP
df["pais_origen_norm"] = (
    df["País de Origen"]
    .fillna("")
    .str.strip()
    .str.lower()
    .map(PAIS_MAP)
)

inst_por_pais = (
    df[df["pais_origen_norm"].notna()]
    .drop_duplicates(subset=["pais_origen_norm", "institucion"])
    .groupby("pais_origen_norm")["institucion"]
    .nunique()
    .reset_index()
    .rename(columns={"pais_origen_norm": "País", "institucion": "Instituciones distintas"})
)

ISO3_MAP = {
    "México": "MEX", "Ecuador": "ECU", "República Dominicana": "DOM",
    "Perú": "PER", "Colombia": "COL", "Argentina": "ARG", "Chile": "CHL",
    "Bolivia": "BOL", "Venezuela": "VEN", "Uruguay": "URY", "Paraguay": "PRY",
    "Brasil": "BRA", "Costa Rica": "CRI", "Guatemala": "GTM", "Honduras": "HND",
    "El Salvador": "SLV", "Nicaragua": "NIC", "Panamá": "PAN", "Cuba": "CUB",
    "Puerto Rico": "PRI",
}

COORDS_MAP = {
    "MEX": (-102, 24),  "ECU": (-78, -2),   "DOM": (-70, 19),
    "PER": (-76, -10),  "COL": (-74, 4),    "ARG": (-64, -35),
    "CHL": (-71, -35),  "BOL": (-65, -17),  "VEN": (-66, 8),
    "URY": (-56, -33),  "PRY": (-58, -23),  "BRA": (-52, -10),
    "CRI": (-84, 10),   "GTM": (-90, 15),   "HND": (-87, 15),
    "SLV": (-89, 14),   "NIC": (-85, 13),   "PAN": (-80, 9),
    "CUB": (-79, 22),   "PRI": (-66, 18),
}

inst_por_pais["iso3"] = inst_por_pais["País"].map(ISO3_MAP)
inst_por_pais = inst_por_pais.dropna(subset=["iso3"])
inst_por_pais["lon"] = inst_por_pais["iso3"].map(lambda x: COORDS_MAP.get(x, (None,None))[0])
inst_por_pais["lat"] = inst_por_pais["iso3"].map(lambda x: COORDS_MAP.get(x, (None,None))[1])

total_inst_pais = inst_por_pais["Instituciones distintas"].sum()

# ── Asignar categoría de rango ───────────────────────────────────────────────
def categorizar(n):
    if n == 0:   return 0    # gris
    elif n <= 5: return 1    # rosa claro
    elif n <= 10: return 2   # rosa fuerte
    else:         return 3   # azul marino

inst_por_pais["rango"] = inst_por_pais["Instituciones distintas"].apply(categorizar)

# Escala discreta: 0=gris, 1=rosa claro, 2=rosa fuerte, 3=azul marino
COLORSCALE_DISCRETA = [
    [0.00, "#d0d0ce"], [0.25, "#d0d0ce"],   # 0   → gris
    [0.25, "#f0a0c8"], [0.50, "#f0a0c8"],   # 1–5 → rosa claro
    [0.50, "#090d50"], [0.75, "#090d50"],   # 5–10→ rosa fuerte
    [0.75, "#e82788"], [1.00, "#e82788"],   # 10+ → azul marino
]

fig = go.Figure()

# ── Capa 1: coroplético con rangos ───────────────────────────────────────────
fig.add_trace(go.Choropleth(
    locations=inst_por_pais["iso3"],
    z=inst_por_pais["rango"],
    text=inst_por_pais["País"],
    customdata=inst_por_pais["Instituciones distintas"],
    colorscale=COLORSCALE_DISCRETA,
    zmin=0, zmax=3,
    showscale=False,
    marker_line_color="white",
    marker_line_width=0.8,
    hovertemplate="<b>%{text}</b><br>Instituciones: %{customdata}<extra></extra>",
))

# ── Capa 2: número dentro del país ───────────────────────────────────────────
fig.add_trace(go.Scattergeo(
    lon=inst_por_pais["lon"],
    lat=inst_por_pais["lat"],
    mode="text",
    text=inst_por_pais["Instituciones distintas"].astype(str),
    textfont=dict(size=11, color="#555555", family="Helvetica"),
    hoverinfo="skip",
    showlegend=False,
))

# ── Leyenda manual ────────────────────────────────────────────────────────────


fig.update_layout(
    title=dict(
        text=f"Instituciones distintas por país  ·  Total: {total_inst_pais}",
        font=dict(size=16, color="#444444", family="Helvetica"),
        x=0.5, xanchor="center", y=0.97,
    ),
    legend=dict(
        title=dict(text="Instituciones", font=dict(size=11, color="#555")),
        orientation="v",
        x=0.01, y=0.35,
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="#dddddd",
        borderwidth=1,
        font=dict(size=10, color="#444", family="Helvetica"),
    ),
    geo=dict(
        showframe=False,
        showcoastlines=False,
        showland=True,  landcolor="#e8e8e8",
        showocean=True, oceancolor="#f0f4fa",
        showlakes=False,
        showcountries=True, countrycolor="white",
        projection_type="mercator",
        lataxis_range=[-60, 38],
        lonaxis_range=[-125, -25],
        bgcolor="#f0f4fa",
    ),
    paper_bgcolor="white",
    height=680,
    margin=dict(l=0, r=0, t=45, b=0),
)

fig.show()

In [31]:
cantidad_pais_tipo = (
    df.groupby(["pais", "tipo_institucion"])["Cantidad"]
    .sum()
    .reset_index()
    .rename(columns={
        "pais": "País",
        "tipo_institucion": "Tipo de institución",
        "Cantidad": "Total preguntas aportadas"
    })
    .sort_values(["País", "Total preguntas aportadas"], ascending=[True, False])
)

print("=" * 65)
print("  TOTAL DE PREGUNTAS POR PAÍS Y TIPO DE INSTITUCIÓN")
print("=" * 65)
# Imprimir agrupado por país para mejor legibilidad
for pais, grupo in cantidad_pais_tipo.groupby("País"):
    print(f"\n── {pais} ──")
    print(grupo[["Tipo de institución", "Total preguntas aportadas"]].to_string(index=False))
    print(f"   Subtotal: {grupo['Total preguntas aportadas'].sum()}")

  TOTAL DE PREGUNTAS POR PAÍS Y TIPO DE INSTITUCIÓN

── Argentina ──
                  Tipo de institución  Total preguntas aportadas
                    Universidad / IES                         43
                      ONG / Fundacion                          7
Centro de Investigacion / Laboratorio                          2
                        Independiente                          1
                                 Otra                          1
   Subtotal: 54

── Bolivia ──
Tipo de institución  Total preguntas aportadas
  Universidad / IES                          2
      Independiente                          0
   Subtotal: 2

── Brasil ──
                  Tipo de institución  Total preguntas aportadas
Centro de Investigacion / Laboratorio                        139
                    Universidad / IES                         77
   Subtotal: 216

── Chile ──
                  Tipo de institución  Total preguntas aportadas
Centro de Investigacion / Laboratorio             

In [32]:
pivot = cantidad_pais_tipo.pivot(
    index="País",
    columns="Tipo de institución",
    values="Total preguntas aportadas"
).fillna(0)

fig = px.imshow(
    pivot,
    title="Total de preguntas por país y tipo de institución",
    labels={"color": "Preguntas"},
    color_continuous_scale=["#C4D8F7", "#4A86D8"],
    text_auto=True,
    aspect="auto"
)
fig.update_layout(
    height=600, paper_bgcolor="white",
    title_font_size=18, xaxis_tickangle=-30
)
fig.show()